# Polygon Type Setup

This notebook sets up the data framework for polygon type analysis, then runs the
Bayesian-style assignment algorithm to fill every empty `County_Type` cell.

**Stage 5 polygon data** is split into 5 location-type DataFrames matching the
community taxonomy defined in `csv_input/Info_CSV/Community_matrix.csv`:

| Variable | Location Type | Transition Matrix File |
|---|---|---|
| `df_inner_city` | Inner City | `Capital_InnerCity.csv` |
| `df_outer_city` | Outer City | `Capital_OuterCity.csv` |
| `df_rural` | Rural | `Capital_Rural.csv` |
| `df_suburb` | Suburb | `Capital_Suburb.csv` |
| `df_town` | Town | `Capital_Town.csv` |

In [ ]:
import sys, os
import pandas as pd
from shapely import wkt
from shapely.strtree import STRtree

# Import random_no_generation as a proper module so CSV_FILE stays in its own namespace
sys.path.insert(0, os.path.abspath('..'))
import random_no_generation as randomnogen

# Patch CSV_FILE to the correct absolute path (module lives one level up, notebook does not)
randomnogen.CSV_FILE = os.path.abspath('../randomno.csv')
print(f'CSV_FILE : {randomnogen.CSV_FILE}')
print(f'Exists   : {os.path.exists(randomnogen.CSV_FILE)}')

# Alias functions into the notebook namespace so the rest of the code is unchanged
return_ran                     = randomnogen.return_ran
return_ran_array               = randomnogen.return_ran_array
random_normal                  = randomnogen.random_normal
random_normal_from_variance    = randomnogen.random_normal_from_variance
random_normal_adv              = randomnogen.random_normal_adv
random_normal_adv_from_variance = randomnogen.random_normal_adv_from_variance
clean_index                    = randomnogen.clean_index

# Smoke-test
print(f'Test random int: {return_ran()}')

In [2]:
# Load main stage 5 polygon dataset
target_df = pd.read_csv('./csv_input/stage5_synced_polygons.csv')

print(f'Total polygons: {len(target_df)}')
print(f'Columns: {list(target_df.columns)}')
print('\nLocation type counts:')
print(target_df['Location'].value_counts())

Total polygons: 1709
Columns: ['County', 'Shape_ID', 'geometry', 'Area', 'Population_Density', 'Region', 'Location', 'County_Type']

Location type counts:
Location
Inner City    892
Outer City    512
Suburb        121
Rural          92
Town           92
Name: count, dtype: int64


In [3]:
# Split into 5 sub-DataFrames by location type
# Names match the community taxonomy in Community_matrix.csv

df_inner_city = target_df[target_df['Location'] == 'Inner City'].reset_index(drop=True)
df_outer_city = target_df[target_df['Location'] == 'Outer City'].reset_index(drop=True)
df_rural      = target_df[target_df['Location'] == 'Rural'].reset_index(drop=True)
df_suburb     = target_df[target_df['Location'] == 'Suburb'].reset_index(drop=True)
df_town       = target_df[target_df['Location'] == 'Town'].reset_index(drop=True)

print('Polygon counts by location type:')
print(f'  Inner City : {len(df_inner_city)}')
print(f'  Outer City : {len(df_outer_city)}')
print(f'  Rural      : {len(df_rural)}')
print(f'  Suburb     : {len(df_suburb)}')
print(f'  Town       : {len(df_town)}')
total = (len(df_inner_city) + len(df_outer_city) + len(df_rural)
         + len(df_suburb) + len(df_town))
print(f'  Total      : {total}')

Polygon counts by location type:
  Inner City : 892
  Outer City : 512
  Rural      : 92
  Suburb     : 121
  Town       : 92
  Total      : 1709


In [4]:
# Load all Transition Matrix files
# Keys match the Location type names from Community_matrix.csv

TRANSITION_MATRIX_DIR = './csv_input/Info_CSV/Transition_Matrix'

transition_matrix_files = {
    'Inner City': 'Capital_InnerCity.csv',
    'Outer City': 'Capital_OuterCity.csv',
    'Rural'     : 'Capital_Rural.csv',
    'Suburb'    : 'Capital_Suburb.csv',
    'Town'      : 'Capital_Town.csv',
}

transition_matrices = {
    location: pd.read_csv(
        os.path.join(TRANSITION_MATRIX_DIR, filename),
        index_col='FROM_TO'
    )
    for location, filename in transition_matrix_files.items()
}

print('Transition matrices loaded:')
for location, tm in transition_matrices.items():
    print(f'  {location:12s}: {tm.shape[0]} x {tm.shape[1]}  columns={list(tm.columns)}')

Transition matrices loaded:
  Inner City  : 14 x 14  columns=['S', 'A1', 'A2', 'B1', 'C1', 'C2', 'D1', 'D2', 'E1', 'E2', 'F1', 'F2', 'TECH', 'UNI']
  Outer City  : 12 x 12  columns=['A1', 'A2', 'B1', 'C1', 'C2', 'D1', 'D2', 'E1', 'E2', 'F1', 'F2', 'UNI']
  Rural       : 5 x 5  columns=['A1', 'C1', 'D1', 'D2', 'F1']
  Suburb      : 7 x 7  columns=['A1', 'B1', 'C1', 'C2', 'E2', 'F1', 'UNI']
  Town        : 6 x 6  columns=['B2', 'C1', 'C2', 'D2', 'E1', 'F1']


## County_Type Assignment Algorithm

Fills every empty `County_Type` cell in `work_df` (a copy of `target_df`).

**Steps per iteration:**

1. `unchecked_list` — all Shape_IDs with empty / NA `County_Type`
2. Draw `return_ran()` (int in \[1, 10 000 000\]) → pick one ID from `unchecked_list`
3. **Prior** — equal weight over all valid types for `(Region, Location)` from `Community_matrix.csv`
4. Inspect neighbours via the adjacency graph; collect those already classified
5. **Likelihood**
   - *0 classified neighbours* → use prior directly
   - *≥1 classified neighbours* → look up each neighbour's type in **its own** location's
     transition matrix, average the rows, then blend:
     `80 × neighbour_avg + 20 × prior`, normalised to sum = 1
6. Sample type from likelihood. If the sampled type is **not valid** for the polygon's
   location, walk the fallback chain (downward first, then upward) until a valid type
   is found:

   `S → A1 → A2 → UNI → TECH → B1 → B2 → C1 → C2 → D2 → D1 → E1 → E2 → F1 → F2`

   > Note: B2 (Mixed New Town) only appears in Town location priors; it is absent from
   > all transition matrices so it can only enter the likelihood via the prior component.

7. Assign the resolved type; remove ID from `unchecked_list`; repeat until empty

In [5]:
# Create working copy — target_df stays pristine; all writes go to work_df
work_df = target_df.copy()

# Normalise County_Type: treat empty strings as NaN for consistent checks
work_df['County_Type'] = work_df['County_Type'].replace('', pd.NA)

print(f'Total polygons : {len(work_df)}')
print(f'Already typed  : {work_df["County_Type"].notna().sum()}')
print(f'To assign      : {work_df["County_Type"].isna().sum()}')
print('\nPre-assigned type counts:')
print(work_df['County_Type'].value_counts())

Total polygons : 1709
Already typed  : 327
To assign      : 1382

Pre-assigned type counts:
County_Type
C1     53
UNI    43
D1     43
B1     41
F2     33
D2     22
E2     22
E1     20
F1     19
C2     16
A1     10
A2      5
Name: count, dtype: int64


In [6]:
# Build adjacency graph from polygon geometries
# Two polygons are adjacent if they share a boundary (touches = shared edge/point,
# no interior overlap). Uses a spatial R-tree index for efficiency.

print('Parsing geometries...')
geometries = target_df['geometry'].apply(wkt.loads).tolist()
shape_ids  = target_df['Shape_ID'].tolist()

print('Building spatial index...')
tree = STRtree(geometries)

print('Finding neighbours...')
adjacency = {}
for i, (sid, geom) in enumerate(zip(shape_ids, geometries)):
    # predicate='touches': geometries sharing a boundary but not overlapping
    neighbour_idxs = tree.query(geom, predicate='touches')
    adjacency[sid] = [shape_ids[j] for j in neighbour_idxs if j != i]

total_edges = sum(len(v) for v in adjacency.values())
avg_n = total_edges / len(adjacency) if adjacency else 0
print(f'Adjacency graph built:')
print(f'  Polygons           : {len(adjacency)}')
print(f'  Total directed edges: {total_edges}')
print(f'  Avg neighbours     : {avg_n:.2f}')

Parsing geometries...
Building spatial index...
Finding neighbours...
Adjacency graph built:
  Polygons           : 1709
  Total directed edges: 12648
  Avg neighbours     : 7.40


In [9]:
# Load community taxonomy and build per-(Region, Location) priors
# Each prior assigns equal probability to every valid type code in that cell

community_df = pd.read_csv('./csv_input/Info_CSV/Community_matrix.csv')

# priors[key]          = {type_code: probability}   — equal weight
# valid_types_map[key] = [type_code, ...]            — list of valid codes
# key format: "{Region}_{Location}" with spaces removed, e.g. 'Capital_InnerCity'

priors          = {}
valid_types_map = {}

for (region, location), grp in community_df.groupby(['Region', 'Type']):
    codes = [str(c).strip() for c in grp['Code'].dropna() if str(c).strip()]
    if not codes:
        continue
    key = f"{region}_{location.replace(' ', '')}"
    n   = len(codes)
    priors[key]          = {code: 1 / n for code in codes}
    valid_types_map[key] = codes

print('Priors created:')
for k, v in priors.items():
    print(f'  {k:28s}: {list(v.keys())}')

Priors created:
  Capital_InnerCity           : ['A1', 'A2', 'B1', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2', 'S', 'TECH', 'UNI']
  Capital_OuterCity           : ['A1', 'A2', 'B1', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2', 'UNI']
  Capital_Rural               : ['A1', 'C1', 'D1', 'D2', 'F1']
  Capital_Suburb              : ['A1', 'B1', 'C1', 'C2', 'E2', 'F1', 'UNI']
  Capital_Town                : ['B2', 'C1', 'C2', 'D2', 'E1', 'F1']
  Highland_InnerCity          : ['A1', 'A2', 'B1', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2', 'S']
  Highland_OuterCity          : ['A1', 'A2', 'B1', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2']
  Highland_Rural              : ['A1', 'C1', 'D1', 'D2', 'F1']
  Highland_Suburb             : ['A1', 'B1', 'C1', 'C2', 'E2', 'F1']
  Highland_Town               : ['B2', 'C1', 'C2', 'D2', 'E1', 'F1']
  Lowland_InnerCity           : ['A1', 'A2', 'B1', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2', 'UNI']
  Lowland_OuterCity           : ['A1', 'A

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────────────

# Ordered fallback chain — highest to lowest prestige.
# B2 (Mixed New Town) sits between B1 and C1. It only appears in Town priors
# (not in any transition matrix), so it will only ever be sampled for Town
# polygons where it is already valid — the chain covers any cross-location edge case.
FALLBACK_CHAIN = [
    'S', 'A1', 'A2', 'UNI', 'TECH',
    'B1', 'B2', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2'
]

# Types that are skipped during fallback chain walking.
# They are only used as an absolute last resort when no other valid type exists.
_FALLBACK_SKIP = {'UNI', 'TECH'}

INFLUENCED_WEIGHT = 80   # neighbour signal weight (out of 100)


# ── Helper functions ──────────────────────────────────────────────────────────

def _prior_key(region, location):
    return f"{region}_{location.replace(' ', '')}"


def get_valid_types(region, location):
    """Return the list of valid type codes for a (Region, Location) pair."""
    return valid_types_map.get(_prior_key(region, location), [])


def select_from_distribution(dist):
    """
    Sample one key from a {type: probability} dict using return_ran().
    Maps the raw int [1, 10_000_000] to a uniform [0, 1) value and uses
    the inverse-CDF method to select a bucket.
    """
    types = list(dist.keys())
    probs = list(dist.values())
    total = sum(probs)
    probs = [p / total for p in probs]

    rand_val   = (return_ran() - 1) / (10_000_000 - 1)   # uniform in [0, 1)
    cumulative = 0.0
    for t, p in zip(types, probs):
        cumulative += p
        if rand_val < cumulative:
            return t
    return types[-1]   # floating-point guard


def find_valid_fallback(selected_type, valid_type_list):
    """
    If selected_type is not in valid_type_list, walk FALLBACK_CHAIN to find
    the nearest valid type (downward first, then upward).

    UNI and TECH are skipped in the first pass — they are only accepted as a
    last resort if no other valid type can be found anywhere in the chain.

    For any type not present in FALLBACK_CHAIN (idx = -1), the downward search
    starts from the beginning of the chain (range(0, len)).
    """
    if not valid_type_list:
        return None
    if selected_type in valid_type_list:
        return selected_type

    try:
        idx = FALLBACK_CHAIN.index(selected_type)
    except ValueError:
        idx = -1   # type absent from chain — search the whole chain downward

    # Pass 1: walk the chain excluding UNI and TECH
    for i in range(idx + 1, len(FALLBACK_CHAIN)):
        if FALLBACK_CHAIN[i] in valid_type_list and FALLBACK_CHAIN[i] not in _FALLBACK_SKIP:
            return FALLBACK_CHAIN[i]
    for i in range(idx - 1, -1, -1):
        if FALLBACK_CHAIN[i] in valid_type_list and FALLBACK_CHAIN[i] not in _FALLBACK_SKIP:
            return FALLBACK_CHAIN[i]

    # Pass 2: nothing found without UNI/TECH — allow them as a last resort
    for i in range(idx + 1, len(FALLBACK_CHAIN)):
        if FALLBACK_CHAIN[i] in valid_type_list:
            return FALLBACK_CHAIN[i]
    for i in range(idx - 1, -1, -1):
        if FALLBACK_CHAIN[i] in valid_type_list:
            return FALLBACK_CHAIN[i]

    return valid_type_list[0]   # absolute last resort


def compute_likelihood(region, location, classified_neighbours):
    """
    Blend neighbour transition signal with the location prior.

    Parameters
    ----------
    region, location : str
        Attributes of the target polygon.
    classified_neighbours : list of (type_code, location_str)
        Neighbours that already have a County_Type.
        Each is looked up in its *own* location's transition matrix.

    Returns
    -------
    dict {type_code: probability}  — normalised to sum = 1

    Notes
    -----
    - B2 is not a column in any transition matrix, so it receives 0 from the
      neighbour component. Its prior weight (20 %) is still preserved for Town
      polygons via the prior term.
    - Neighbours whose type is not found in their location's transition matrix
      (e.g. B2 neighbours) are skipped — they contribute nothing to the blend.
    """
    prior = priors.get(_prior_key(region, location), {})

    if not classified_neighbours:
        return dict(prior)

    # Collect all type columns present across all matrices
    all_type_cols = set()
    for tm in transition_matrices.values():
        all_type_cols.update(tm.columns.tolist())

    # Sum transition rows from every classified neighbour
    neighbour_sum = {t: 0.0 for t in all_type_cols}
    count = 0

    for n_type, n_location in classified_neighbours:
        tm = transition_matrices.get(n_location)
        if tm is None or n_type not in tm.index:
            continue   # B2 neighbours (not in TM index) are skipped here
        row = tm.loc[n_type]
        for t in all_type_cols:
            if t in row.index:
                neighbour_sum[t] += float(row[t])
        count += 1

    if count == 0:
        return dict(prior)   # all neighbours were untraceable — fall back to prior

    neighbour_avg = {t: neighbour_sum[t] / count for t in all_type_cols}

    # Blend: influenced_weight * neighbour_avg + (100 - influenced_weight) * prior
    # Union includes B2 for Town polygons (from prior) even though TMs lack it
    all_keys = set(prior.keys()) | all_type_cols
    blended  = {}
    for t in all_keys:
        n_prob = neighbour_avg.get(t, 0.0)   # 0 for B2 (not in TMs)
        p_prob = prior.get(t, 0.0)
        blended[t] = INFLUENCED_WEIGHT * n_prob + (100 - INFLUENCED_WEIGHT) * p_prob

    total = sum(blended.values())
    if total == 0:
        return dict(prior)

    return {t: v / total for t, v in blended.items() if v > 0}


print('Helper functions defined.')
print(f'FALLBACK_CHAIN    : {FALLBACK_CHAIN}')
print(f'FALLBACK_SKIP     : {_FALLBACK_SKIP}')
print(f'INFLUENCED_WEIGHT : {INFLUENCED_WEIGHT}')


In [ ]:
# ── O(1) lookup structures ────────────────────────────────────────────────────
_idx     = work_df.set_index('Shape_ID')
ct_state = _idx['County_Type'].to_dict()
meta     = _idx[['Region', 'Location']].to_dict('index')

# Step 1: build unchecked_list
unchecked_list = [
    sid for sid, ct in ct_state.items()
    if pd.isna(ct) or str(ct).strip() == ''
]

print(f'Polygons to assign : {len(unchecked_list)}')
print(f'Already assigned   : {len(ct_state) - len(unchecked_list)}')

# ── Pre-fetch all random numbers in one CSV read ──────────────────────────────
# Each iteration uses at most 3 numbers (step-2 pick, step-6 sample, rare fallback).
# return_ran_array(n) does exactly ONE read + ONE write of randomno.csv for all n ints.
# Everything after this is fully offline — no further file I/O.

_POOL_SIZE  = len(unchecked_list) * 3
_rand_list  = return_ran_array(_POOL_SIZE)
_rand_iter  = iter(_rand_list)

print(f'Random pool        : {_POOL_SIZE} integers pre-fetched (1 CSV read/write total)')

# Rebind return_ran to draw from the in-memory list so select_from_distribution
# also uses the pool without any code changes in that function.
def return_ran():
    return next(_rand_iter)

# ── Main assignment loop (fully offline) ──────────────────────────────────────
iteration = 0
log_every = max(1, len(unchecked_list) // 10)

while unchecked_list:
    iteration += 1

    # Step 2: random pick from unchecked_list
    rand_no     = return_ran()
    pick_idx    = (rand_no - 1) % len(unchecked_list)
    selected_id = unchecked_list[pick_idx]

    region   = meta[selected_id]['Region']
    location = meta[selected_id]['Location']

    # Step 4: gather classified neighbours from adjacency graph
    classified_neighbours = []
    for n_id in adjacency.get(selected_id, []):
        n_ct = ct_state.get(n_id)
        if n_ct and not pd.isna(n_ct) and str(n_ct).strip():
            classified_neighbours.append((str(n_ct).strip(), meta[n_id]['Location']))

    # Steps 5a / 5b: compute blended likelihood
    likelihood = compute_likelihood(region, location, classified_neighbours)

    # Step 6: sample type from likelihood
    # select_from_distribution calls return_ran() — rebound above, draws from pool
    if likelihood:
        selected_type = select_from_distribution(likelihood)
    else:
        valid = get_valid_types(region, location)
        selected_type = valid[(return_ran() - 1) % len(valid)] if valid else None

    # Step 6 (cont.): validate; walk fallback chain if type not valid for location
    if selected_type:
        valid_for_loc = get_valid_types(region, location)
        if selected_type not in valid_for_loc:
            selected_type = find_valid_fallback(selected_type, valid_for_loc)

    # Step 7: assign
    if selected_type:
        ct_state[selected_id] = selected_type

    # Step 8: remove from unchecked_list and continue
    unchecked_list.pop(pick_idx)

    if iteration % log_every == 0 or not unchecked_list:
        print(f'  [{iteration:>5d}] remaining: {len(unchecked_list)}')

# Write results back to work_df
work_df['County_Type'] = work_df['Shape_ID'].map(ct_state)

print(f'\nCompleted — {iteration} polygons assigned.')
print(f'Unassigned remaining: {work_df["County_Type"].isna().sum()}')
print('\nCounty_Type distribution:')
print(work_df['County_Type'].value_counts())

In [ ]:
# ── Print work_df overview ────────────────────────────────────────────────────
print('work_df (first 10 rows):')
print(work_df[['County', 'Shape_ID', 'Location', 'County_Type']].head(10).to_string())
print(f'\nShape: {work_df.shape}')
print(f'\nCounty_Type value counts:\n{work_df["County_Type"].value_counts()}')

# ── Column order: follow FALLBACK_CHAIN hierarchy (left = highest prestige) ───
# Only keep types that actually appear in the data
col_order = [t for t in FALLBACK_CHAIN if t in work_df['County_Type'].values]

# ── group_county_work_df ──────────────────────────────────────────────────────
# Rows = County, Columns = County_Type (hierarchy order), Values = % within county

county_counts = (
    work_df
    .groupby(['County', 'County_Type'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=col_order, fill_value=0)
)

group_county_work_df = (
    county_counts
    .div(county_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

print('\ngroup_county_work_df  (County_Type % per County, left→right = high→low prestige):')
print(group_county_work_df.to_string())

# ── group_district_work_df ────────────────────────────────────────────────────
# County code split into letter prefix + numeric suffix  e.g. "NC1" → "NC" | "1"
# Group only by the letter prefix (district)

work_df['_district'] = work_df['County'].str.extract(r'^([A-Za-z]+)')

district_counts = (
    work_df
    .groupby(['_district', 'County_Type'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=col_order, fill_value=0)
)

group_district_work_df = (
    district_counts
    .div(district_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)
group_district_work_df.index.name = 'District'

print('\ngroup_district_work_df  (County_Type % per District, left→right = high→low prestige):')
print(group_district_work_df.to_string())

# Clean up temporary column
work_df.drop(columns=['_district'], inplace=True)